# Avazu Click-Through Rate Prediction 


<img src='https://md.teyit.org/img/tiklama-direnci-kapak.png'>


Bu çalışmada `Avazu Click-Through Rate Prediction` yarışması için reklam gösterim verileri kullanılarak bir reklama tıklanma olasılığını tahmin eden bir sınıflandırma modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. ROC AUC değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde reklam tıklama tahmini için gerekli veri hazırlama ve sınıflandırma kütüphanelerini içe aktarıyoruz.


In [2]:
from google.colab import drive
import os
import zipfile
import gzip

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [ ]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/avazu-ctr-prediction.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, 'train.gz')
test_member = find_zip_member(zip_members, 'test.gz')
sample_submission_member = find_zip_member(zip_members, 'sampleSubmission.gz')


## 3. Veri Dosyalarını Yükleme


In [6]:
sample_train_rows = 500000
sample_test_rows = 200000

selected_columns = [
    'id', 'click', 'hour', 'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category',
    'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip', 'device_model',
    'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21'
]

def read_gzip_csv_from_zip(zip_path, member_name, nrows=None, usecols=None):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as raw_file:
            with gzip.open(raw_file, 'rt') as gz_file:
                return pd.read_csv(gz_file, nrows=nrows, usecols=usecols)

train = read_gzip_csv_from_zip(zip_path, train_member, nrows=sample_train_rows, usecols=selected_columns)
test_usecols = [col for col in selected_columns if col != 'click']
test = read_gzip_csv_from_zip(zip_path, test_member, nrows=sample_test_rows, usecols=test_usecols)
sample_submission = read_gzip_csv_from_zip(zip_path, sample_submission_member)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape (sample): (500000, 24)
Test shape (sample): (200000, 23)
Sample submission shape: (4577464, 2)


,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,app_category,device_id,device_ip,device_model,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21
0,1.000009e+18,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,ddd2926e,44956a24,1,2,15706,320,50,1722,0,35,-1,79
1,1.000017e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,96809ac8,711ee120,1,0,15704,320,50,1722,0,35,100084,79
2,1.000037e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,b3cf8def,8a4875bd,1,0,15704,320,50,1722,0,35,100084,79
3,1.000064e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,e8275b8f,6332421a,1,0,15706,320,50,1722,0,35,100084,79
4,1.000068e+19,0,14102100,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,07d7df22,a99f214a,9644d0bf,779d90c2,1,0,18993,320,50,2161,0,35,-1,157


## 4. Ön İşleme


In [7]:
# Bu bölümde eksik olabilecek alanları kontrol ediyor ve tarih bilgisini daha kullanışlı hale getiriyoruz.


In [8]:
train_df = train.copy()
test_df = test.copy()

for df in [train_df, test_df]:
    df['hour'] = df['hour'].astype(str)
    df['day'] = df['hour'].str[4:6]
    df['hour_of_day'] = df['hour'].str[6:8]

print('Train null count:', train_df.isnull().sum().sum())
print('Test null count:', test_df.isnull().sum().sum())
train_df.head()


Train null count: 0
Test null count: 0


,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,app_category,device_id,device_ip,device_model,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21,day,hour_of_day
0,1.000009e+18,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,ddd2926e,44956a24,1,2,15706,320,50,1722,0,35,-1,79,21,00
1,1.000017e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,96809ac8,711ee120,1,0,15704,320,50,1722,0,35,100084,79,21,00
2,1.000037e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,b3cf8def,8a4875bd,1,0,15704,320,50,1722,0,35,100084,79,21,00
3,1.000064e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,e8275b8f,6332421a,1,0,15706,320,50,1722,0,35,100084,79,21,00
4,1.000068e+19,0,14102100,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,07d7df22,a99f214a,9644d0bf,779d90c2,1,0,18993,320,50,2161,0,35,-1,157,21,00


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde model girdilerini hazırlıyor ve kategorik alanları belirliyoruz.


In [10]:
feature_columns = [
    'hour', 'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category',
    'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip', 'device_model',
    'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21',
    'day', 'hour_of_day'
]

x = train_df[feature_columns].copy()
y = train_df['click'].copy()
test_x = test_df[feature_columns].copy()

for col in x.columns:
    x[col] = x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = feature_columns
numerical_features = []

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (400000, 24)
x_valid shape: (100000, 24)


## 6. Model Kurma


In [11]:
# Bu bölümde tüm alanları kategorik olarak işleyip tıklama olasılığı tahmini yapan başlangıç modelini kuruyoruz.


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=120,
        max_depth=16,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample'
    ))
])

model.fit(x_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['hour', 'C1', 'banner_pos',
                                                   'site_id', 'site_domain',
                                                   'site_category', 'app_id',
                                                   'app_domain', 'app_category',
                                                   'device_id', 'device_ip',
                                                   'device_model',
                                                   'device_type',
                                                   'device_conn_type', 'C14',
                                                   'C15', 'C16', 'C17', 'C18',
                                                   'C19', 'C20', 'C21', 'day',
                                                   'hour_of_day'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=16, min_samples_split=10,
                                        n_estimators=120, n_jobs=-1,
                                        random_state=42))])

## 7. ROC AUC Değerlendirmesi


In [13]:
# Bu bölümde modelin reklam tıklamasını ayırt etme gücünü ROC AUC ile ölçüyoruz.


In [14]:
valid_probs = model.predict_proba(x_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_probs)
print('Validation ROC AUC:', round(roc_auc, 5))


Validation ROC AUC: 0.71736


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test verisi için tıklanma olasılıklarını üretip submission dosyasını oluşturuyoruz.


In [16]:
test_probs = model.predict_proba(test_x)[:, 1]

submission = pd.DataFrame({
    'id': test_df['id'].values,
    'click': test_probs
})

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (200000, 2)


,id,click
0,10000174058809263569,0.495763
1,10000182526920855428,0.506313
2,10000554139829213984,0.507781
3,10001094637809798845,0.443479
4,10001377041558670745,0.487697


In [17]:
output_path = '/content/avazu_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/avazu_submission.csv


## 9. Sonuç


Bu çalışmada Avazu Click-Through Rate Prediction yarışması için reklam gösterim verileri kullanılarak bir reklama tıklanma olasılığını tahmin eden bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.71736 ROC AUC değeri elde etti ve test verisi için tıklanma olasılıkları üretildi.
